# STOCKER — GPU Training on Colab

Bu notebook:
1. GitHub'dan repo'yu clone'lar
2. Bağımlılıkları kurar
3. GPU kontrolü yapar
4. Veri toplar (S&P 500 OHLCV + haberler)
5. Dataset build eder (feature engineering + labeling)
6. 4 model + meta-learner eğitir
7. Sonuçları Google Drive'a kaydeder

**Runtime > Change runtime type > GPU (T4)** seçili olmalı!

## 0. Config

In [ ]:
# ── Config ──────────────────────────────────────────────
GITHUB_REPO = "beratkaskaloglu/stocker"
BRANCH = "main"
MARKET = "US"
EPOCHS = 100
BATCH_SIZE = 64
SAVE_TO_DRIVE = True  # True = sonuclari Google Drive'a kaydet

## 1. GPU Kontrol & Setup

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("\n\u26a0\ufe0f GPU yok! Runtime > Change runtime type > GPU secin.")

## 2. Repo Clone & Dependencies

In [ ]:
import os

# Clone repo
if not os.path.exists("/content/stocker"):
    !git clone --depth 1 -b {BRANCH} https://github.com/{GITHUB_REPO}.git /content/stocker
else:
    !cd /content/stocker && git pull origin {BRANCH}

os.chdir("/content/stocker")
print(f"\nCWD: {os.getcwd()}")

In [ ]:
# Install dependencies (Colab already has torch, numpy, pandas)
!pip install -q -r requirements.txt 2>&1 | tail -5
print("\nDependencies installed.")

## 3. Google Drive Mount (opsiyonel)

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    
    DRIVE_DIR = "/content/drive/MyDrive/stocker"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
    print(f"Drive directory: {DRIVE_DIR}")
else:
    DRIVE_DIR = None
    print("Drive mount skipped.")

## 4. Veri Toplama (S&P 500 OHLCV + Haberler)

In [ ]:
# Once Drive'da hazir veri var mi kontrol et
DATASET_DIR = "/content/stocker/data/sp500_dataset"
OHLCV_CSV = "/content/stocker/data/ohlcv_sp500.csv"
NEWS_JSON = "/content/stocker/data/news_sp500.json"

import shutil
from pathlib import Path

# Eger Drive'da hazir dataset varsa kopyala
if DRIVE_DIR and Path(f"{DRIVE_DIR}/data/sp500_dataset/X.npy.dat").exists():
    print("Drive'da hazir dataset bulundu, kopyalaniyor...")
    if not Path(DATASET_DIR).exists():
        shutil.copytree(f"{DRIVE_DIR}/data/sp500_dataset", DATASET_DIR)
    print(f"Dataset: {DATASET_DIR}")
    SKIP_DATA_COLLECTION = True
elif DRIVE_DIR and Path(f"{DRIVE_DIR}/data/ohlcv_sp500.csv").exists():
    print("Drive'da ham veri bulundu, kopyalaniyor...")
    shutil.copy2(f"{DRIVE_DIR}/data/ohlcv_sp500.csv", OHLCV_CSV)
    if Path(f"{DRIVE_DIR}/data/news_sp500.json").exists():
        shutil.copy2(f"{DRIVE_DIR}/data/news_sp500.json", NEWS_JSON)
    SKIP_DATA_COLLECTION = True
else:
    SKIP_DATA_COLLECTION = False
    print("Veri toplanacak...")

In [ ]:
if not SKIP_DATA_COLLECTION:
    import yfinance as yf
    import pandas as pd
    from datetime import datetime, timedelta

    # S&P 500 sembol listesi (top 100 by market cap)
    SP500_TOP = [
        "AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "META", "BRK-B", "TSLA", "UNH", "LLY",
        "JPM", "XOM", "V", "JNJ", "PG", "MA", "AVGO", "HD", "MRK", "COST",
        "ABBV", "CVX", "CRM", "KO", "PEP", "ADBE", "WMT", "BAC", "MCD", "TMO",
        "CSCO", "ACN", "ABT", "NFLX", "LIN", "AMD", "DHR", "ORCL", "INTC", "DIS",
        "PM", "TXN", "WFC", "INTU", "CMCSA", "VZ", "COP", "NEE", "AMGN", "RTX",
        "HON", "UPS", "IBM", "LOW", "SPGI", "GE", "CAT", "BA", "ELV", "QCOM",
        "PLD", "AMAT", "SBUX", "DE", "NOW", "MS", "GS", "BLK", "SYK", "MDLZ",
        "ADP", "LMT", "ISRG", "MMC", "GILD", "TJX", "BKNG", "ADI", "VRTX", "CB",
        "CI", "REGN", "AMT", "SCHW", "PGR", "LRCX", "MU", "TMUS", "ZTS", "SO",
        "BSX", "FI", "KLAC", "SNPS", "CME", "EOG", "EQIX", "ITW", "SHW", "APD"
    ]

    END_DATE = datetime.now().strftime("%Y-%m-%d")
    START_DATE = (datetime.now() - timedelta(days=365*5)).strftime("%Y-%m-%d")

    print(f"Downloading {len(SP500_TOP)} symbols: {START_DATE} -> {END_DATE}")

    all_dfs = []
    failed = []

    for i, sym in enumerate(SP500_TOP):
        try:
            df = yf.download(sym, start=START_DATE, end=END_DATE, progress=False)
            if len(df) > 100:
                df = df.reset_index()
                df.columns = [c.lower() for c in df.columns]
                df = df.rename(columns={"adj close": "close"})  # use adjusted close
                df["symbol"] = sym
                df = df[["date", "open", "high", "low", "close", "volume", "symbol"]]
                all_dfs.append(df)
            if (i + 1) % 20 == 0:
                print(f"  [{i+1}/{len(SP500_TOP)}] downloaded")
        except Exception as e:
            failed.append(sym)

    ohlcv = pd.concat(all_dfs, ignore_index=True)
    os.makedirs("data", exist_ok=True)
    ohlcv.to_csv(OHLCV_CSV, index=False)
    print(f"\nSaved: {OHLCV_CSV} ({len(ohlcv)} rows, {len(all_dfs)} symbols)")
    if failed:
        print(f"Failed: {failed}")

    # Drive'a yedekle
    if DRIVE_DIR:
        shutil.copy2(OHLCV_CSV, f"{DRIVE_DIR}/data/ohlcv_sp500.csv")
        print(f"Backed up to Drive.")

## 5. Dataset Build (Feature Engineering + Labeling)

In [ ]:
from pathlib import Path

# Dataset zaten varsa atla
if Path(DATASET_DIR).exists() and Path(f"{DATASET_DIR}/X.npy.dat").exists():
    print(f"Dataset zaten mevcut: {DATASET_DIR}")
    import numpy as np
    shape = np.load(f"{DATASET_DIR}/X_shape.npy")
    print(f"  Shape: ({int(shape[0])}, {int(shape[1])}, {int(shape[2])})")
else:
    print("Building dataset (feature engineering + labeling)...")
    print("Bu islem 15-30 dakika surebilir (100 sembol x 5 yil).")
    
    !python scripts/build_dataset.py \
        --ohlcv {OHLCV_CSV} \
        --out {DATASET_DIR} \
        --seq-len 60 \
        --threshold 0.01 \
        --no-finbert  # Colab'da ilk calistirmada FinBERT'siz hizli build

    # Drive'a yedekle
    if DRIVE_DIR:
        import shutil
        if Path(f"{DRIVE_DIR}/data/sp500_dataset").exists():
            shutil.rmtree(f"{DRIVE_DIR}/data/sp500_dataset")
        shutil.copytree(DATASET_DIR, f"{DRIVE_DIR}/data/sp500_dataset")
        print(f"Dataset backed up to Drive.")

## 6. Model Training (GPU)

In [ ]:
# Training
from core.training.train_all import train_all_models

summary = train_all_models(
    market=MARKET,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    device="cuda",
    data_path=DATASET_DIR,
)

In [ ]:
# Sonuclari goster
import json

print("\n" + "=" * 60)
print("TRAINING RESULTS")
print("=" * 60)
print(f"Market: {summary['market']}")
print(f"Horizons: {summary['horizons']}")
print(f"Total time: {summary['total_time_minutes']:.1f} min")
print()

for name, result in summary["models"].items():
    print(f"  {name:20s} | val_loss={result['best_val_loss']:.4f} | epochs={result['epochs_trained']}")

print(f"\nModels saved to: outputs/models/{MARKET}/")

## 7. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/logs/{MARKET}

## 8. Sonuclari Drive'a Kaydet

In [ ]:
if DRIVE_DIR:
    import shutil
    from pathlib import Path

    # Model checkpoint'lari
    src_models = Path(f"outputs/models/{MARKET}")
    dst_models = Path(f"{DRIVE_DIR}/models/{MARKET}")
    if src_models.exists():
        if dst_models.exists():
            shutil.rmtree(dst_models)
        shutil.copytree(src_models, dst_models)
        print(f"Models saved to Drive: {dst_models}")

    # Training logs
    src_logs = Path(f"outputs/logs/{MARKET}")
    dst_logs = Path(f"{DRIVE_DIR}/logs/{MARKET}")
    if src_logs.exists():
        if dst_logs.exists():
            shutil.rmtree(dst_logs)
        shutil.copytree(src_logs, dst_logs)
        print(f"Logs saved to Drive: {dst_logs}")

    # Summary
    summary_path = src_models / "training_summary.json"
    if summary_path.exists():
        shutil.copy2(summary_path, f"{DRIVE_DIR}/training_summary.json")

    total_size = sum(f.stat().st_size for f in dst_models.rglob("*") if f.is_file()) / 1e6
    print(f"\nTotal size on Drive: {total_size:.1f} MB")
    print("\nColab kapansa bile modeller Drive'da guvende!")
else:
    print("Drive mount edilmedi. Modelleri manuel indirin:")
    print("  Sol panelde outputs/models/ klasorune gidin > sag tik > Download")

## 9. Modeli Lokal'e Geri Cekme

Egitim bittikten sonra, lokal makinede:

```bash
# Google Drive'dan (Drive sync ile otomatik)
cp -r ~/Google\ Drive/stocker/models/ outputs/models/

# Ya da Colab'dan direkt download ettiysen:
# outputs/models/ klasorunu projeye kopyala
```